# Customer Churn Prediction

Predicting telecom customer churn from contract, service, and billing data.

**Result:** CatBoost achieved a test ROC-AUC of 0.914 and accuracy of 0.925.

**Methods:** classification, feature engineering, CatBoost, LightGBM, cross-validation, ROC analysis.

> This portfolio version removes course-review correspondence and repetitive instructional text. The analysis, models, and reported metrics are based on the original completed project. The source datasets are not included in this repository.


## 1. Setup and data loading

The four source tables contain contract, personal, internet-service, and phone-service data. They are merged by customer ID before modelling.


In [4]:
import copy
import datetime
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sweetviz as sv

from IPython.display import display

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import cv, CatBoostClassifier
from sklearn.metrics import (roc_auc_score,
                             accuracy_score,
                             roc_curve)
from sklearn.model_selection import (train_test_split,
                                     GridSearchCV,
                                     cross_val_score,
                                     cross_val_predict)
from sklearn.preprocessing import (OneHotEncoder,
                                    OrdinalEncoder,
                                    StandardScaler)
import pandas_profiling
import phik


INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


/var/folders/mz/b2hbqf51521_fzm8c30chvf40000gn/T/ipykernel_7922/2079936165.py:31: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  import pandas_profiling


In [5]:
try:
    df_contract = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/14/final_provider/contract_new.csv')
    df_personal = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/14/final_provider/personal_new.csv')
    df_internet = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/14/final_provider/internet_new.csv')
    df_phone = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/14/final_provider/phone_new.csv')

except:
    df_contract = pd.read_csv(
        'https://code.s3.yandex.net/datasets/contract_new.csv')
    df_personal = pd.read_csv(
        'https://code.s3.yandex.net/datasets/personal_new.csv')
    df_internet = pd.read_csv(
        'https://code.s3.yandex.net/datasets/internet_new.csv')
    df_phone = pd.read_csv(
        'https://code.s3.yandex.net/datasets/phone_new.csv')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [6]:
# report = sv.analyze([df_contract, "Info_df_contract_1.0"])
# report.show_html('Info_df_contract_1.0.html')
# report.show_notebook('Info_df_contract_1.0.html')


In [7]:
# report = sv.analyze([df_personal, "Info_df_personal_1.0"])
# report.show_html('Info_df_personal_1.0.html')
# report.show_notebook('Info_df_personal_1.0.html')


In [8]:
# report = sv.analyze([df_internet, "Info_df_internet_1.0"])
# report.show_html('Info_df_internet_1.0.html')
# report.show_notebook('Info_df_internet_1.0.html')


In [9]:
#report = sv.analyze([df_phone, "Info_df_phone_1.0"])
# report.show_html('Info_df_phone_1.0.html')
# report.show_notebook('Info_df_phone_1.0.html')


## 2. Data integration and cleaning

This section merges the tables, standardises column names, resolves missing service records, creates the churn target, and removes non-predictive identifiers.


In [10]:
df_list = [df_contract, df_personal, df_internet, df_phone]
for i in df_list:
    display(i.head(10))
    print(i.info())
    print()
    print(i.shape)
    print()
    print(i.dtypes)


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,31.04
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,2071.84
2,3668-QPYBK,2019-10-01,No,Month-to-month,Yes,Mailed check,53.85,226.17
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1960.6
4,9237-HQITU,2019-09-01,No,Month-to-month,Yes,Electronic check,70.70,353.5
5,9305-CDSKC,2019-03-01,No,Month-to-month,Yes,Electronic check,99.65,1150.96
6,1452-KIOVK,2018-04-01,No,Month-to-month,Yes,Credit card (automatic),89.10,2058.21
7,6713-OKOMC,2019-04-01,No,Month-to-month,No,Mailed check,29.75,300.48
8,7892-POOKP,2017-07-01,No,Month-to-month,Yes,Electronic check,104.80,3573.68
9,6388-TABGU,2014-12-01,2017-05-01,One year,No,Bank transfer (automatic),56.15,1628.35


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB
None

(7043, 8)

customerID           object
BeginDate            object
EndDate              object
Type                 object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No
5,9305-CDSKC,Female,0,No,No
6,1452-KIOVK,Male,0,No,Yes
7,6713-OKOMC,Female,0,No,No
8,7892-POOKP,Female,0,Yes,No
9,6388-TABGU,Male,0,No,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-null   object
 4   Dependents     7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB
None

(7043, 5)

customerID       object
gender           object
SeniorCitizen     int64
Partner          object
Dependents       object
dtype: object


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No
5,9305-CDSKC,Fiber optic,No,No,Yes,No,Yes,Yes
6,1452-KIOVK,Fiber optic,No,Yes,No,No,Yes,No
7,6713-OKOMC,DSL,Yes,No,No,No,No,No
8,7892-POOKP,Fiber optic,No,No,Yes,Yes,Yes,Yes
9,6388-TABGU,DSL,Yes,Yes,No,No,No,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB
None

(5517, 8)

customerID          object
InternetService     object
OnlineSecurity      object
OnlineBackup        object
DeviceProtection    object
TechSupport         object
StreamingTV         object
StreamingMovies     object
dtype: object


,customerID,MultipleLines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes
5,7892-POOKP,Yes
6,6388-TABGU,No
7,9763-GRSKD,No
8,7469-LKBCI,No
9,8091-TTVAX,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6361 entries, 0 to 6360
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     6361 non-null   object
 1   MultipleLines  6361 non-null   object
dtypes: object(2)
memory usage: 99.5+ KB
None

(6361, 2)

customerID       object
MultipleLines    object
dtype: object


In [11]:
df_merge = pd.merge(df_contract, df_personal, how='left', on='customerID')
df_merge = pd.merge(df_merge, df_internet, how='left', on='customerID')
df_merge = pd.merge(df_merge, df_phone, how='left', on='customerID')


In [12]:
df_merge.columns = df_merge.columns.str.replace(
    r"([A-Z])", r" \1").str.lower().str.replace(' ', '_').str[1:]

print(df_merge.shape)
print()

print(df_merge.dtypes)


(7043, 20)

ustomer_i_d           object
begin_date            object
end_date              object
type                  object
paperless_billing     object
payment_method        object
monthly_charges      float64
total_charges         object
ender                 object
senior_citizen         int64
partner               object
dependents            object
internet_service      object
online_security       object
online_backup         object
device_protection     object
tech_support          object
streaming_t_v         object
streaming_movies      object
multiple_lines        object
dtype: object


/var/folders/mz/b2hbqf51521_fzm8c30chvf40000gn/T/ipykernel_7922/2417915822.py:1: FutureWarning: The default value of regex will change from True to False in a future version.
  df_merge.columns = df_merge.columns.str.replace(


In [13]:
df_merge = df_merge.rename(columns={'ustomer_i_d': 'customer_id',
                                    'streaming_t_v': 'streaming_tv',
                                    'ender': 'gender'}
                           )

print(df_merge.dtypes)


customer_id           object
begin_date            object
end_date              object
type                  object
paperless_billing     object
payment_method        object
monthly_charges      float64
total_charges         object
gender                object
senior_citizen         int64
partner               object
dependents            object
internet_service      object
online_security       object
online_backup         object
device_protection     object
tech_support          object
streaming_tv          object
streaming_movies      object
multiple_lines        object
dtype: object


In [14]:
print(df_merge.duplicated().sum())


0


In [15]:
pd.DataFrame(df_merge.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [16]:
df_merge = df_merge.fillna('No')


In [17]:
pd.DataFrame(df_merge.isna().mean()
             .to_frame(name='Missing share')
             .query('Missing share > 0')['Missing share'])\
    .style.background_gradient('coolwarm')\
    .format({'Missing share': '{:.2%}'})


In [18]:
display(df_merge.head(10))


,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,gender,senior_citizen,partner,dependents,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,multiple_lines
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,31.04,Female,0,Yes,No,DSL,No,Yes,No,No,No,No,No
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,2071.84,Male,0,No,No,DSL,Yes,No,Yes,No,No,No,No
2,3668-QPYBK,2019-10-01,No,Month-to-month,Yes,Mailed check,53.85,226.17,Male,0,No,No,DSL,Yes,Yes,No,No,No,No,No
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1960.6,Male,0,No,No,DSL,Yes,No,Yes,Yes,No,No,No
4,9237-HQITU,2019-09-01,No,Month-to-month,Yes,Electronic check,70.70,353.5,Female,0,No,No,Fiber optic,No,No,No,No,No,No,No
5,9305-CDSKC,2019-03-01,No,Month-to-month,Yes,Electronic check,99.65,1150.96,Female,0,No,No,Fiber optic,No,No,Yes,No,Yes,Yes,Yes
6,1452-KIOVK,2018-04-01,No,Month-to-month,Yes,Credit card (automatic),89.10,2058.21,Male,0,No,Yes,Fiber optic,No,Yes,No,No,Yes,No,Yes
7,6713-OKOMC,2019-04-01,No,Month-to-month,No,Mailed check,29.75,300.48,Female,0,No,No,DSL,Yes,No,No,No,No,No,No
8,7892-POOKP,2017-07-01,No,Month-to-month,Yes,Electronic check,104.80,3573.68,Female,0,Yes,No,Fiber optic,No,No,Yes,Yes,Yes,Yes,Yes
9,6388-TABGU,2014-12-01,2017-05-01,One year,No,Bank transfer (automatic),56.15,1628.35,Male,0,No,Yes,DSL,Yes,Yes,No,No,No,No,No


In [19]:
# display(df_merge.head(10))


In [20]:
df_merge['exited'] = df_merge['end_date'].apply(
    lambda x: 0 if x == 'No' else 1)


In [21]:
# display(df_merge.head(10))


In [22]:
df_merge['end_date'] = df_merge['end_date'].replace('No', '2020-02-01')


In [23]:
df_merge['end_date'] = pd.to_datetime(df_merge['end_date'])
df_merge['begin_date'] = pd.to_datetime(df_merge['begin_date'])
df_merge['duration'] = (
    (df_merge['end_date'] - df_merge['begin_date']) / np.timedelta64(1, 'D')).round()
df_merge['duration'] = df_merge['duration'].astype(int)


In [24]:
df_merge = df_merge.drop(columns=['begin_date',
                                  'end_date']
                         )


In [25]:
df_merge = df_merge.set_index('customer_id')


In [26]:
display(df_merge.info())


<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   type               7043 non-null   object 
 1   paperless_billing  7043 non-null   object 
 2   payment_method     7043 non-null   object 
 3   monthly_charges    7043 non-null   float64
 4   total_charges      7043 non-null   object 
 5   gender             7043 non-null   object 
 6   senior_citizen     7043 non-null   int64  
 7   partner            7043 non-null   object 
 8   dependents         7043 non-null   object 
 9   internet_service   7043 non-null   object 
 10  online_security    7043 non-null   object 
 11  online_backup      7043 non-null   object 
 12  device_protection  7043 non-null   object 
 13  tech_support       7043 non-null   object 
 14  streaming_tv       7043 non-null   object 
 15  streaming_movies   7043 non-null   object 
 16  multiple_lines

None

In [27]:
display(df_merge.head(10))


,type,paperless_billing,payment_method,monthly_charges,total_charges,gender,senior_citizen,partner,dependents,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,multiple_lines,exited,duration
customer_id,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Month-to-month,Yes,Electronic check,29.85,31.04,Female,0,Yes,No,DSL,No,Yes,No,No,No,No,No,0,31
5575-GNVDE,One year,No,Mailed check,56.95,2071.84,Male,0,No,No,DSL,Yes,No,Yes,No,No,No,No,0,1036
3668-QPYBK,Month-to-month,Yes,Mailed check,53.85,226.17,Male,0,No,No,DSL,Yes,Yes,No,No,No,No,No,0,123
7795-CFOCW,One year,No,Bank transfer (automatic),42.30,1960.6,Male,0,No,No,DSL,Yes,No,Yes,Yes,No,No,No,0,1371
9237-HQITU,Month-to-month,Yes,Electronic check,70.70,353.5,Female,0,No,No,Fiber optic,No,No,No,No,No,No,No,0,153
9305-CDSKC,Month-to-month,Yes,Electronic check,99.65,1150.96,Female,0,No,No,Fiber optic,No,No,Yes,No,Yes,Yes,Yes,0,337
1452-KIOVK,Month-to-month,Yes,Credit card (automatic),89.10,2058.21,Male,0,No,Yes,Fiber optic,No,Yes,No,No,Yes,No,Yes,0,671
6713-OKOMC,Month-to-month,No,Mailed check,29.75,300.48,Female,0,No,No,DSL,Yes,No,No,No,No,No,No,0,306
7892-POOKP,Month-to-month,Yes,Electronic check,104.80,3573.68,Female,0,Yes,No,Fiber optic,No,No,Yes,Yes,Yes,Yes,Yes,0,945


In [28]:
def unique_values(df):
    for column in df.columns:
        print(f"Unique values in {column}:")
        print(df_merge[column].unique())
        print("")


print(unique_values(df_merge))


Unique values in type:
['Month-to-month' 'One year' 'Two year']

Unique values in paperless_billing:
['Yes' 'No']

Unique values in payment_method:
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Unique values in monthly_charges:
[29.85 56.95 53.85 ... 63.1  44.2  78.7 ]

Unique values in total_charges:
['31.04' '2071.84' '226.17' ... '325.6' '520.8' '7251.82']

Unique values in gender:
['Female' 'Male']

Unique values in senior_citizen:
[0 1]

Unique values in partner:
['Yes' 'No']

Unique values in dependents:
['No' 'Yes']

Unique values in internet_service:
['DSL' 'Fiber optic' 'No']

Unique values in online_security:
['No' 'Yes']

Unique values in online_backup:
['Yes' 'No']

Unique values in device_protection:
['No' 'Yes']

Unique values in tech_support:
['No' 'Yes']

Unique values in streaming_tv:
['No' 'Yes']

Unique values in streaming_movies:
['No' 'Yes']

Unique values in multiple_lines:
['No' 'Yes']

Unique values in exited:
[0 1]


In [29]:
unique_counts = df_merge.nunique()
print(unique_counts)


type                    3
paperless_billing       2
payment_method          4
monthly_charges      1585
total_charges        6658
gender                  2
senior_citizen          2
partner                 2
dependents              2
internet_service        3
online_security         2
online_backup           2
device_protection       2
tech_support            2
streaming_tv            2
streaming_movies        2
multiple_lines          2
exited                  2
duration              251
dtype: int64


In [30]:
display(df_merge.info())


<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   type               7043 non-null   object 
 1   paperless_billing  7043 non-null   object 
 2   payment_method     7043 non-null   object 
 3   monthly_charges    7043 non-null   float64
 4   total_charges      7043 non-null   object 
 5   gender             7043 non-null   object 
 6   senior_citizen     7043 non-null   int64  
 7   partner            7043 non-null   object 
 8   dependents         7043 non-null   object 
 9   internet_service   7043 non-null   object 
 10  online_security    7043 non-null   object 
 11  online_backup      7043 non-null   object 
 12  device_protection  7043 non-null   object 
 13  tech_support       7043 non-null   object 
 14  streaming_tv       7043 non-null   object 
 15  streaming_movies   7043 non-null   object 
 16  multiple_lines

None

In [31]:
unique_values_t_c = df_merge["total_charges"].unique()
unique_values_sorted_t_c = sorted(unique_values_t_c)
print(unique_values_sorted_t_c)


[' ', '100.17', '100.19', '100.2', '100.24', '100.25', '100.4', '100.75', '100.88', '100.9', '100.94', '1000.0', '1000.12', '1000.43', '1001.0', '1001.1', '1001.81', '1002.05', '1002.13', '1002.46', '1003.15', '1003.93', '1004.08', '1005.2', '1005.3', '1006.0', '1006.95', '1007.1', '1007.5', '1008.37', '1008.45', '1008.48', '1008.58', '1008.6', '1008.8', '1009.65', '1009.8', '101.0', '101.2', '101.25', '101.75', '1010.1', '1010.65', '1011.66', '1012.35', '1012.5', '1013.27', '1013.5', '1014.75', '1015.07', '1015.2', '1016.6', '1017.45', '1017.9', '1018.37', '1018.77', '1018.8', '1019.4', '102.01', '102.25', '102.38', '102.41', '102.61', '102.64', '102.75', '102.8', '1020.6', '1021.76', '1021.8', '1023.51', '1023.8', '1023.85', '1024.06', '1024.1', '1027.31', '1028.98', '1028.99', '103.25', '103.52', '103.55', '103.84', '1030.26', '1031.05', '1032.2', '1032.35', '1033.6', '1033.85', '1034.28', '1035.34', '1035.94', '1036.15', '1036.2', '1036.59', '1037.7', '1037.95', '1038.6', '1039.35'

In [32]:
df_merge.query("total_charges == ' '")


,type,paperless_billing,payment_method,monthly_charges,total_charges,gender,senior_citizen,partner,dependents,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,multiple_lines,exited,duration
customer_id,,,,,,,,,,,,,,,,,,,
4472-LVYGI,Two year,Yes,Bank transfer (automatic),52.55,,Female,0,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,No,No,0,0
3115-CZMZD,Two year,No,Mailed check,20.25,,Male,0,No,Yes,No,No,No,No,No,No,No,No,0,0
5709-LVOEQ,Two year,No,Mailed check,80.85,,Female,0,Yes,Yes,DSL,Yes,Yes,Yes,No,Yes,Yes,No,0,0
4367-NUYAO,Two year,No,Mailed check,25.75,,Male,0,Yes,Yes,No,No,No,No,No,No,No,Yes,0,0
1371-DWPAZ,Two year,No,Credit card (automatic),56.05,,Female,0,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,No,No,0,0
7644-OMVMY,Two year,No,Mailed check,19.85,,Male,0,Yes,Yes,No,No,No,No,No,No,No,No,0,0
3213-VVOLG,Two year,No,Mailed check,25.35,,Male,0,Yes,Yes,No,No,No,No,No,No,No,Yes,0,0
2520-SGTTA,Two year,No,Mailed check,20.00,,Female,0,Yes,Yes,No,No,No,No,No,No,No,No,0,0
2923-ARZLG,One year,Yes,Mailed check,19.70,,Male,0,Yes,Yes,No,No,No,No,No,No,No,No,0,0


In [33]:
count_space = (df_merge['total_charges'] == " ").sum()
print(count_space)


11


In [34]:
count_zero = (df_merge['duration'] == 0).sum()
print(count_zero)


11


In [35]:
df_merge = df_merge.drop(df_merge[df_merge['duration'] == 0].index)


In [36]:
df_merge['total_charges'] = df_merge['total_charges'].astype('float64')


In [37]:
df_merge['senior_citizen'] = df_merge['senior_citizen'].astype('object')


In [38]:
display(df_merge.info())


<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   type               7032 non-null   object 
 1   paperless_billing  7032 non-null   object 
 2   payment_method     7032 non-null   object 
 3   monthly_charges    7032 non-null   float64
 4   total_charges      7032 non-null   float64
 5   gender             7032 non-null   object 
 6   senior_citizen     7032 non-null   object 
 7   partner            7032 non-null   object 
 8   dependents         7032 non-null   object 
 9   internet_service   7032 non-null   object 
 10  online_security    7032 non-null   object 
 11  online_backup      7032 non-null   object 
 12  device_protection  7032 non-null   object 
 13  tech_support       7032 non-null   object 
 14  streaming_tv       7032 non-null   object 
 15  streaming_movies   7032 non-null   object 
 16  multiple_lines

None

## 3. Exploratory analysis

Numerical distributions, class patterns, and associations between mixed data types are examined before feature selection.


In [39]:
df_numeric = df_merge[['monthly_charges',
                       'duration',
                       'total_charges']
                      ]


In [40]:
df_numeric.describe()


,monthly_charges,duration,total_charges
count,7032.000000,7032.000000,7032.000000
mean,64.798208,899.961320,2118.621822
std,30.085974,682.738777,2112.736199
min,18.250000,28.000000,19.050000
25%,35.587500,276.000000,439.745000
50%,70.350000,761.000000,1345.275000
75%,89.862500,1461.000000,3239.317500
max,118.750000,2314.000000,9221.380000


In [41]:
fig, axs = plt.subplots(2, 2, figsize=(15, 15))
axs = axs.flatten()

for i, col in enumerate(df_numeric.columns):
    axs[i].hist(df_numeric[col], bins=20)
    axs[i].set_title(col)

for ax in axs:
    ax.title.set_fontsize(18)

plt.tight_layout()
plt.show()
plt.close()


<Figure size 1080x1080 with 4 Axes>

In [42]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(
    12, 8), gridspec_kw={'wspace': 1.3})

df_numeric.boxplot(column=['duration'], ax=axes[0], grid=True)
axes[0].set_title('Contract duration', y=1.03, fontsize=20)
axes[0].set_ylabel('Contract duration, days', fontsize=16)
axes[0].set_xlabel('')

df_numeric.boxplot(column=['monthly_charges'], ax=axes[1], grid=True)
axes[1].set_title('Monthly charges', y=1.03, fontsize=20)
axes[1].set_ylabel('Monthly charges, $', fontsize=16)
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()
plt.close()


/var/folders/mz/b2hbqf51521_fzm8c30chvf40000gn/T/ipykernel_7922/3893971940.py:16: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


<Figure size 864x576 with 2 Axes>

In [43]:
df_numeric = df_merge[['monthly_charges',
                       'exited',
                       'duration',
                       'total_charges']
                      ]


In [44]:
fig, ax = plt.subplots(figsize=(18, 11))
sns.histplot(df_numeric['monthly_charges'], bins=30,
             kde=True, color='blue', alpha=0.5, ax=ax)
ax.set_title('Distribution of monthly charges', y=1.03, fontsize=20)
ax.set_xlabel('Monthly charge', fontsize=16)
ax.set_ylabel('Customer count', fontsize=16)
plt.show()
plt.close()
print()

fig, ax = plt.subplots(figsize=(12, 8))
sns.boxplot(x='exited', y='duration', data=df_numeric, ax=ax)
ax.set_title(
    'Contract duration by churn status', y=1.03, fontsize=20)
ax.set_xlabel('Churn status', fontsize=16)
ax.set_ylabel('Duration, months', fontsize=16)
plt.show()
plt.close()


<Figure size 1296x792 with 1 Axes>

<Figure size 864x576 with 1 Axes>

In [45]:
phik_overview = df_merge.drop(['monthly_charges',
                               'total_charges',
                               'senior_citizen',
                               'duration'], axis=1).phik_matrix()
phik_overview['exited'].sort_values(ascending=False)


interval columns not set, guessing: ['exited']


exited               1.000000
multiple_lines       0.261345
online_backup        0.229736
partner              0.227597
streaming_movies     0.221756
device_protection    0.218622
payment_method       0.214300
streaming_tv         0.200334
online_security      0.132914
tech_support         0.103933
type                 0.094622
paperless_billing    0.082789
internet_service     0.056279
dependents           0.048710
gender               0.008800
Name: exited, dtype: float64

In [46]:
df_numeric = df_merge[['monthly_charges',
                       'total_charges',
                       'duration']
                      ]


In [47]:
sns.set(font_scale=1.15)
plt.figure(figsize=(15, 4))
sns.heatmap(
    df_numeric.corr(),
    cmap='RdBu_r',
    annot=True,
    vmin=-1, vmax=1)
plt.show()
plt.close()


<Figure size 1080x288 with 2 Axes>

In [48]:
df_numeric = df_merge[['monthly_charges',
                       'exited',
                       'duration',
                       'total_charges']
                      ]


In [49]:
g = sns.pairplot(df_numeric,
                 hue='exited',
                 diag_kind='kde',
                 diag_kws=dict(shade=True),
                 height=5,
                 palette=['#52BE80',
                          '#FF5733'],
                 plot_kws=dict(s=30,
                               linewidth=0.5)
                 )

g.fig.subplots_adjust(top=0.95)
g.fig.suptitle('Relationships between numerical features',
               fontsize=23,
               y=1.0
               )

g.axes[1, 0].set_xlabel('Monthly Charges', fontsize=16)
g.axes[1, 0].set_ylabel('Total Charges', fontsize=16)

g.axes[2, 0].set_xlabel('Monthly Charges', fontsize=16)
g.axes[2, 0].set_ylabel('Duration', fontsize=16)

g.axes[2, 1].set_xlabel('Total Charges', fontsize=16)
g.axes[2, 1].set_ylabel('Duration', fontsize=16)

plt.show()
plt.close()


/Users/anaconda/anaconda3/lib/python3.9/site-packages/seaborn/axisgrid.py:1507: FutureWarning: 

`shade` is now deprecated in favor of `fill`; setting `fill=True`.
This will become an error in seaborn v0.14.0; please update your code.

  func(x=vector, **plot_kwargs)
/Users/anaconda/anaconda3/lib/python3.9/site-packages/seaborn/axisgrid.py:1507: FutureWarning: 

`shade` is now deprecated in favor of `fill`; setting `fill=True`.
This will become an error in seaborn v0.14.0; please update your code.

  func(x=vector, **plot_kwargs)
/Users/anaconda/anaconda3/lib/python3.9/site-packages/seaborn/axisgrid.py:1507: FutureWarning: 

`shade` is now deprecated in favor of `fill`; setting `fill=True`.
This will become an error in seaborn v0.14.0; please update your code.

  func(x=vector, **plot_kwargs)


<Figure size 1132.54x1080 with 12 Axes>

In [50]:
df_numeric = df_merge[['monthly_charges',
                       'exited',
                       'duration',
                       'senior_citizen',
                       'total_charges']
                      ]


In [51]:
sns.set(font_scale=1.5)
plt.figure(figsize=(22, 10))
sns.heatmap(
    df_merge.phik_matrix(interval_cols=df_numeric),
    cmap='RdBu_r',
    annot=True,
    annot_kws={"size": 14},
    vmin=-1, vmax=1)
plt.title("Phi-k association between categorical features",
          fontsize=23, y=1.03)
plt.show()
plt.close()


<Figure size 1584x720 with 2 Axes>

In [52]:
df_merge = df_merge.drop('total_charges', axis=1)


In [53]:
print(df_merge.duplicated().sum())


22


In [54]:
df_merge = df_merge.drop_duplicates()


In [55]:
# report = sv.analyze([df_merge, "Info_df_merge_1.0"])
# report.show_html('Info_df_merge_1.0.html')
# report.show_notebook('Info_df_merge_1.0.html')


In [56]:
# pandas_profiling.ProfileReport(df_merge)


## 4. Feature preparation

The data are split before encoding. Categorical variables are one-hot encoded using transformations fitted on the training sample.


In [57]:
features = df_merge.drop(['exited'], axis=1)
target = df_merge['exited']

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=100423)


In [58]:
ohe_features = features_train.select_dtypes(include='object').columns.to_list()


In [59]:
num_features = features_train.select_dtypes(exclude='object').columns.to_list()


In [60]:
features_train_ohe = features_train.copy()

features_test_ohe = features_test.copy()


In [61]:
(features_train_ohe.columns != features_test_ohe.columns).sum()


0

In [62]:
display(df_merge.columns)


Index(['type', 'paperless_billing', 'payment_method', 'monthly_charges',
       'gender', 'senior_citizen', 'partner', 'dependents', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'multiple_lines', 'exited',
       'duration'],
      dtype='object')

In [63]:
encoder_ohe = OneHotEncoder(
    drop='first', handle_unknown='ignore', sparse=False)

encoder_ohe.fit(features_train_ohe[ohe_features])

features_train_ohe[
    encoder_ohe.get_feature_names_out()
] = encoder_ohe.transform(features_train_ohe[ohe_features])

features_train_ohe = features_train_ohe.drop(ohe_features, axis=1)

scaler = StandardScaler()

features_train_ohe[num_features] = scaler.fit_transform(
    features_train_ohe[num_features])

features_train_ohe.head()


,monthly_charges,duration,type_One year,type_Two year,paperless_billing_Yes,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,gender_Male,senior_citizen_1,partner_Yes,dependents_Yes,internet_service_Fiber optic,internet_service_No,online_security_Yes,online_backup_Yes,device_protection_Yes,tech_support_Yes,streaming_tv_Yes,streaming_movies_Yes,multiple_lines_Yes
customer_id,,,,,,,,,,,,,,,,,,,,,
1559-DTODC,-1.326197,-0.472603,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2812-SFXMJ,-1.495452,-1.277979,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9283-LZQOH,0.308275,-0.653042,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9572-MTILT,1.381883,-0.119059,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
9475-NNDGC,1.594281,1.849637,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0


In [64]:
features_test_ohe[
    encoder_ohe.get_feature_names_out()
] = encoder_ohe.transform(features_test_ohe[ohe_features])

features_test_ohe = features_test_ohe.drop(ohe_features, axis=1)

features_test_ohe[num_features] = scaler.transform(
    features_test_ohe[num_features]
)

features_test_ohe.head()


,monthly_charges,duration,type_One year,type_Two year,paperless_billing_Yes,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,gender_Male,senior_citizen_1,partner_Yes,dependents_Yes,internet_service_Fiber optic,internet_service_No,online_security_Yes,online_backup_Yes,device_protection_Yes,tech_support_Yes,streaming_tv_Yes,streaming_movies_Yes,multiple_lines_Yes
customer_id,,,,,,,,,,,,,,,,,,,,,
2833-SLKDQ,-0.665771,-1.143016,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8436-BJUMM,0.618576,-0.117592,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4709-LKHYG,-1.497112,-0.028106,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9907-SWKKF,-1.329516,-1.143016,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9681-OXGVC,1.174462,-1.008053,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0


In [65]:
features_train_ohe.shape, features_test_ohe.shape


((5257, 21), (1753, 21))

In [66]:
(features_train_ohe.columns != features_test_ohe.columns).sum()


0

In [67]:
display(features_train_ohe.columns)


Index(['monthly_charges', 'duration', 'type_One year', 'type_Two year',
       'paperless_billing_Yes', 'payment_method_Credit card (automatic)',
       'payment_method_Electronic check', 'payment_method_Mailed check',
       'gender_Male', 'senior_citizen_1', 'partner_Yes', 'dependents_Yes',
       'internet_service_Fiber optic', 'internet_service_No',
       'online_security_Yes', 'online_backup_Yes', 'device_protection_Yes',
       'tech_support_Yes', 'streaming_tv_Yes', 'streaming_movies_Yes',
       'multiple_lines_Yes'],
      dtype='object')

In [68]:

#encoder = OrdinalEncoder()

#cat_columns = ['type',
#               'paperless_billing',
#               'payment_method',
#               'gender',
#               'partner',
#               'dependents',
#               'internet_service',
#               'online_security',
#               'online_backup',
#               'device_protection',
#               'tech_support',
#               'streaming_tv',
#               'streaming_movies',
#               'multiple_lines',
#               'senior_citizen'
#               ]

#cat_features = features[cat_columns]

#cat_features = pd.DataFrame(encoder.fit_transform(cat_features),
#                            columns=cat_features.columns,
#                            index=cat_features.index
#                            )

#ordinal_features = features.copy()

#for column in cat_columns:
#    ordinal_features[column] = cat_features[column]


In [69]:
#ordinal_features_train = features_train.copy()
#ordinal_features_test = features_test.copy()

#ordinal_features_train = ordinal_features.loc[ordinal_features_train.index, :]
#ordinal_features_test = ordinal_features.loc[ordinal_features_test.index, :]


In [70]:

#(ordinal_features_train.columns != ordinal_features_test.columns).sum()


## 5. Model selection

Random Forest, CatBoost, and LightGBM models are compared with cross-validation using ROC-AUC as the primary metric.


In [71]:
# %%time

grid_params_RFC = {'max_depth': np.arange(1, 110, 50),
                   'max_features': ['log2', 'sqrt'],
                   'min_samples_leaf': [x for x in range(2, 10, 5)],
                   'n_estimators': [x for x in range(500, 810, 100)],
                   'class_weight': ['balanced', None]
                   }

model_forest = RandomForestClassifier(random_state=100423)

print('# Tuning hyper-parameters for ROC-AUC')
print()
grid_RFC = GridSearchCV(model_forest,
                        param_grid=grid_params_RFC,
                        cv=5,
                        verbose=1,
                        n_jobs=-1,
                        scoring='roc_auc'
                        )


grid_RFC.fit(features_train_ohe, target_train)
print("Best parameters set found on development set:")
print()
print(grid_RFC.best_params_)
print()

#best_model = grid_RFC.best_estimator_
#best_model.fit(features_train_ohe, target_train)
#preds = best_model.predict_proba(features_train_ohe)[:, 1]
#cv_roc_auc_RFC = roc_auc_score(target_train, preds)

cv_roc_auc_RFC = grid_RFC.best_score_

print("ROC_AUC for 'RandomForestClassifier' with tuned parameters :",
      round(cv_roc_auc_RFC, 3))


In [72]:
#best_model = grid_RFC.best_estimator_

#y_pred = best_model.predict(features_train_ohe)

#accuracy = accuracy_score(target_train, y_pred)


In [73]:
best_model = grid_RFC.best_estimator_

cv_scores = cross_val_score(best_model, features_train_ohe, target_train, cv=5)

print("Cross-validation accuracy scores for 'RandomForestClassifier':", cv_scores)
print()
print("Mean accuracy for 'RandomForestClassifier':", round(cv_scores.mean(), 3))


In [74]:
grid_params_CBC = {'learning_rate': np.arange(0.01, 0.2, 0.05),
                   'depth': np.arange(2, 15, 3),
                   'random_state': [100423],
                   'verbose': [False]
                   }

model_CBC = CatBoostClassifier(random_state=100423,
                               loss_function='CrossEntropy',
                               iterations=200
                               )

print('# Tuning hyper-parameters for ROC-AUC')
print()
grid_CBC = GridSearchCV(model_CBC,
                        param_grid=grid_params_CBC,
                        cv=5,
                        verbose=0,
                        n_jobs=-1,
                        scoring='roc_auc'
                        )


grid_CBC.fit(features_train_ohe, target_train)
print("Best parameters set found on development set:")
print()
print(grid_CBC.best_params_)
print()

cv_roc_auc_CBC = grid_CBC.best_score_

print("ROC_AUC for 'CatBoostClassifier[encoded features]' with tuned parameters :", round(
    cv_roc_auc_CBC, 3))


In [75]:
#best_model = grid_CBC.best_estimator_

#y_pred = best_model.predict(features_train_ohe)

#accuracy = accuracy_score(target_train, y_pred)


In [76]:
best_model = grid_CBC.best_estimator_

cv_scores = cross_val_score(best_model, features_train_ohe, target_train, cv=5)

print(
    "Cross-validation accuracy scores for 'CatBoostClassifier[encoded features]':", cv_scores)
print()
print("Mean accuracy for 'CatBoostClassifier[encoded features]':", round(
    cv_scores.mean(), 3))


In [77]:
grid_params_LGBMC = {'num_leaves': [7, 15, 31],
                     'learning_rate': np.arange(0.10, 0.21, 0.05),
                     'n_estimators': [50, 100, 200],
                     'min_child_samples': [5, 10, 20],
                     'reg_alpha': [0, 0.1, 0.5],
                     'reg_lambda': [0, 0.1, 0.5],
                     'random_state': [100423]
                     }

model_LGBMC = LGBMClassifier()

print('# Tuning hyper-parameters for ROC-AUC')
print()
grid_LGBMC = GridSearchCV(model_LGBMC,
                          param_grid=grid_params_LGBMC,
                          cv=5,
                          n_jobs=-1,
                          verbose=0,
                          scoring='roc_auc'
                          )

grid_LGBMC.fit(features_train_ohe, target_train)
print("Best parameters set found on development set:")
print()
print(grid_LGBMC.best_params_)
print()

cv_roc_auc_LGBMR = grid_LGBMC.best_score_

print("ROC-AUC for 'LightGBMClassifier' with tuned parameters on the training set:",
      round(cv_roc_auc_LGBMR, 3))


# Tuning hyper-parameters for ROC-AUC



/Users/anaconda/anaconda3/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:702: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [78]:
#best_model = grid_LGBMC.best_estimator_

#y_pred = best_model.predict(features_train_ohe)

#accuracy = accuracy_score(target_train, y_pred)


In [79]:
best_model = grid_LGBMC.best_estimator_

cv_scores = cross_val_score(best_model, features_train_ohe, target_train, cv=5)

print("Cross-validation accuracy scores for 'LightGBMClassifier':", cv_scores)
print()
print("Mean accuracy for 'LightGBMClassifier':", round(cv_scores.mean(), 3))


In [80]:
plt.figure(figsize=[12, 9])

plt.plot([0, 1], [0, 1], linestyle='--', label='RandomModel')

RFC_probs = cross_val_predict(grid_RFC.best_estimator_, features_train_ohe,
                              target_train, cv=5, method='predict_proba')[:, 1]
fpr, tpr, thresholds = roc_curve(target_train, RFC_probs)
auc_roc_RFC_train = roc_auc_score(target_train, RFC_probs)
accuracy_RFC_train = accuracy_score(
    target_train, grid_RFC.predict(features_train_ohe))
plt.plot(fpr, tpr, label='RandomForestClassifier_train')

CBC_probs = cross_val_predict(grid_CBC.best_estimator_, features_train_ohe,
                              target_train, cv=5, method='predict_proba')[:, 1]
fpr, tpr, thresholds = roc_curve(target_train, CBC_probs)
auc_roc_CBC_train = roc_auc_score(target_train, CBC_probs)
accuracy_CBC_train = accuracy_score(
    target_train, grid_CBC.predict(features_train_ohe))
plt.plot(fpr, tpr, label='CatBoostClassifier_train')

LGBMC_probs = cross_val_predict(
    grid_LGBMC.best_estimator_, features_train_ohe, target_train, cv=5, method='predict_proba')[:, 1]
fpr, tpr, thresholds = roc_curve(target_train, LGBMC_probs)
auc_roc_LGBMC_train = roc_auc_score(target_train, LGBMC_probs)
accuracy_LGBMC_train = accuracy_score(
    target_train, grid_LGBMC.predict(features_train_ohe))
plt.plot(fpr, tpr, label='LightGBMClassifier_train')


plt.xlim([0, 1])
plt.ylim([0, 1])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend(loc='lower right', fontsize=14)

plt.title("ROC curve", fontsize=23, y=1.02)
plt.show()
plt.close()


<Figure size 864x648 with 1 Axes>

## 6. Final evaluation

The selected CatBoost model is evaluated once on the held-out test set. The ROC curve and feature importance provide additional diagnostic context.


In [81]:
model_CBC_test = CatBoostClassifier(learning_rate=0.16,
                                    depth=5,
                                    verbose=False,
                                    random_state=100423
                                    )

model_CBC_test.fit(features_train_ohe, target_train)


predict_proba = model_CBC_test.predict_proba(features_test_ohe)[:, 1]

ROC_AUC = round((roc_auc_score(target_test, predict_proba)), 3)
print('ROC_AUC for "CatBoostClassifier" on the test set:', ROC_AUC)


In [82]:
y_pred = model_CBC_test.predict(features_test_ohe)

accuracy = accuracy_score(target_test, y_pred)
print("Accuracy for 'CatBoostClassifier' :", round(accuracy, 3))


In [83]:
plt.figure(figsize=[12,9])

plt.plot([0, 1], [0, 1], linestyle='--', label='RandomModel')

probabilities_test = grid_CBC.best_estimator_.predict_proba(features_test_ohe)[:, 1]
probabilities_one_test = probabilities_test
fpr, tpr, thresholds = roc_curve(target_test, probabilities_one_test)
auc_roc_CBC_test = roc_auc_score(target_test, probabilities_test)
accuracy_CBC_test = accuracy_score(target_test, grid_CBC.predict(features_test_ohe))
plt.plot(fpr, tpr, label='CatBoostClassifier_test')

plt.xlim([0,1])
plt.ylim([0,1])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend(loc='lower right', fontsize=14)

plt.title("ROC curve", fontsize=23, y=1.02)
plt.show()
plt.close()


<Figure size 864x648 with 1 Axes>

In [84]:
feat_import = pd.DataFrame(
    data={'feature': features_train_ohe.columns, 'percent': model_CBC_test.feature_importances_})
feat_import_sorted = feat_import.sort_values('percent', ascending=False).reset_index(drop=True)
display(feat_import_sorted)

plt.figure(figsize=(13, 8))
plt.bar(x=feat_import_sorted['feature'], height=feat_import_sorted['percent'])
plt.xticks(rotation=90)
plt.xlabel('Features')
plt.ylabel('Importance, %')
plt.title('Feature Importance')
plt.show()
plt.close()


,feature,percent
0,duration,47.137626
1,monthly_charges,17.192497
2,partner_Yes,4.023399
3,type_Two year,3.892385
4,multiple_lines_Yes,2.766791
5,online_backup_Yes,2.335236
6,payment_method_Credit card (automatic),2.127021
7,device_protection_Yes,2.120247
8,streaming_movies_Yes,2.082632
9,type_One year,2.036334


<Figure size 936x576 with 1 Axes>

## Conclusion

The final model separates likely churners well on held-out data. Customer tenure is the strongest predictor in the fitted model; contract and service attributes also contribute. In an operational setting, decision thresholds should be chosen from retention cost, intervention capacity, and the relative cost of missed churners.
